In [15]:
import pandas as pd

import plotly.express as px

import irina.utility_functions as uf

In [5]:
pd.read_csv(uf.PATH+f"df_jaccard_dist_yearly_collaborative.csv")

,year,country,from_start,diag
0,1970,AE,0.000000,0.000000
1,1971,AE,0.916667,0.916667
2,1972,AE,0.900000,1.000000
3,1973,AE,0.818182,0.800000
4,1974,AE,1.000000,0.800000
...,...,...,...,...
9918,2019,TL,1.000000,0.852941
9919,2020,TL,1.000000,0.774194
9920,2021,TL,1.000000,0.789474
9921,2022,TL,1.000000,0.770833


In [38]:
dist_types = ["w1", "jaccard"]
modes = ["individual", "collaborative"]

df_dist_years_list = []
for dist in dist_types:
    for mode in modes:
        df_dist_years_list.append(
            pd.read_csv(uf.PATH+f"df_{dist}_dist_yearly_{mode}.csv").rename(columns={"index": "year"}).assign(
                mode=mode, dist=dist)
        )
df_dist_years = pd.concat(df_dist_years_list)

In [39]:
df_dist_years

,year,country,from_start,diag,mode,dist
0,1970,AE,0.000000,0.000000,individual,w1
1,1971,AE,0.491786,0.491786,individual,w1
2,1972,AE,0.511223,0.127980,individual,w1
3,1973,AE,0.320180,0.241239,individual,w1
4,1974,AE,0.505034,0.231754,individual,w1
...,...,...,...,...,...,...
9918,2019,TL,1.000000,0.852941,collaborative,jaccard
9919,2020,TL,1.000000,0.774194,collaborative,jaccard
9920,2021,TL,1.000000,0.789474,collaborative,jaccard
9921,2022,TL,1.000000,0.770833,collaborative,jaccard


In [42]:
country_list = df_dist_years["country"].drop_duplicates().sample(5)
country_list = ["US", "CN", "ID", "IN"]
px.line(
    (
        df_dist_years
        .query("country in @country_list")
    ),
    x="year",
    y="diag",
    color="country",
    hover_data=["mode", "dist"],
)

In [50]:
df_stats = pd.read_csv(uf.PATH+"df_country_stats_yearly.csv")

In [51]:
df_stats.head()

,country,mode,year,entropy,norm_entropy,total_articles,log_total_articles,gini
0,AE,individual,1970,2.740875,0.496045,34.0,3.526361,0.323529
1,AF,individual,1970,1.609438,0.291277,5.0,1.609438,0.000000
2,AM,individual,1970,3.126757,0.565882,65.0,4.174387,0.383333
3,AO,individual,1970,0.693147,0.125446,2.0,0.693147,0.000000
4,AR,individual,1970,4.278805,0.774381,873.0,6.771936,0.600798


In [52]:
country = "AE"
df_stats.query("country == @country")

,country,mode,year,entropy,norm_entropy,total_articles,log_total_articles,gini
0,AE,individual,1970,2.740875,0.496045,34.0,3.526361,0.323529
173,AE,collaborative,1970,2.163956,0.400201,10.0,2.302585,0.088889
324,AE,individual,1971,2.273212,0.411407,29.0,3.367296,0.397878
489,AE,collaborative,1971,1.386294,0.257902,4.0,1.386294,0.000000
616,AE,individual,1972,2.771928,0.501665,31.0,3.433987,0.288625
...,...,...,...,...,...,...,...,...
19089,AE,collaborative,2020,4.614476,0.834530,5936.0,8.688791,0.647890
19306,AE,individual,2021,4.636234,0.838465,3923.0,8.274612,0.630564
19521,AE,collaborative,2021,4.645021,0.840054,7247.0,8.888343,0.642084
19738,AE,individual,2022,4.641234,0.839369,4503.0,8.412499,0.625381


In [54]:
df_dist_stats = (
    df_dist_years
    .merge(df_stats, on=["year", "country", "mode"])
)

In [66]:
df_dist_stats.query("dist == \"w1\"")[["diag", "from_start", "total_articles", "log_total_articles", "gini"]].corr()

,diag,from_start,total_articles,log_total_articles,gini
diag,1.000000,0.500572,-0.121365,-0.657533,-0.665799
from_start,0.500572,1.000000,-0.124394,-0.473308,-0.456892
total_articles,-0.121365,-0.124394,1.000000,0.364879,0.216590
log_total_articles,-0.657533,-0.473308,0.364879,1.000000,0.930733
gini,-0.665799,-0.456892,0.216590,0.930733,1.000000


In [65]:
px.scatter(
    df_dist_stats.query("dist == \"w1\""),
    x="diag",
    y="gini",
    color="mode",
)

In [74]:
df_pct = (
    df_dist_years
    .sort_values(["country", "mode", "dist", "year"])
    .groupby(["country", "mode", "dist"], as_index=False)
    .apply(lambda g: g.assign(
        pct_change=g.diag.pct_change()
    ))
    .reset_index(drop=True)
)

/var/folders/kq/3vlmcmh917x4z481rl4yj6t00000gn/T/ipykernel_1168/151026444.py:2: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [76]:
df_pct

,year,country,from_start,diag,mode,dist,pct_change
0,1993,AD,0.000000,0.000000,collaborative,jaccard,NaN
1,2001,AD,1.000000,1.000000,collaborative,jaccard,inf
2,2002,AD,1.000000,1.000000,collaborative,jaccard,0.000000
3,2003,AD,1.000000,1.000000,collaborative,jaccard,0.000000
4,2004,AD,1.000000,1.000000,collaborative,jaccard,0.000000
...,...,...,...,...,...,...,...
40331,2018,ZW,0.557004,0.073172,individual,w1,-0.112016
40332,2019,ZW,0.570317,0.037844,individual,w1,-0.482800
40333,2020,ZW,0.552649,0.040251,individual,w1,0.063593
40334,2021,ZW,0.579648,0.073581,individual,w1,0.828037


['US',
 'CN',
 'GB',
 'DE',
 'CA',
 'AU',
 'IT',
 'FR',
 'ES',
 'IN',
 'JP',
 'NL',
 'SA',
 'BR',
 'CH',
 'KR',
 'SE',
 'PK',
 'IR',
 'RU',
 'BE',
 'MY',
 'DK',
 'EG',
 'PL',
 'TR',
 'AT',
 'HK',
 'PT',
 'TW',
 'NO',
 'SG',
 'MX',
 'FI',
 'ID',
 'ZA',
 'CZ',
 'IL',
 'IE',
 'GR',
 'CL',
 'NZ',
 'TH',
 'AE',
 'NG',
 'CO',
 'VN',
 'BD',
 'AR',
 'HU',
 'UA',
 'RO',
 'IQ',
 'TN',
 'JO',
 'DZ',
 'QA',
 'MA',
 'RS',
 'SK',
 'MO',
 'ET',
 'EC',
 'KZ',
 'HR',
 'SI',
 'PE',
 'CY',
 'GH',
 'KE',
 'LB',
 'PH',
 'BG',
 'EE',
 'UG',
 'LT',
 'OM',
 'LU',
 'NP',
 'KW',
 'LK',
 'TZ',
 'CM',
 'BY',
 'UY',
 'IS',
 'CR',
 'LV',
 'SD',
 'YE',
 'PR',
 'UZ',
 'CU',
 'AZ',
 'PS',
 'BH',
 'BA',
 'VE',
 'KH',
 'ZW',
 'TJ',
 'BN',
 'ZM',
 'MK',
 'SS',
 'MN',
 'RW',
 'MW',
 'SN',
 'MT',
 'BJ',
 'MZ',
 'BF',
 'PA',
 'GE',
 'AM',
 'CD',
 'CI',
 'BO',
 'BW',
 'LY',
 'SY',
 'AL',
 'KG',
 'AF',
 'PY',
 'GT',
 'FJ',
 'MD',
 'ME',
 'MM',
 'ML',
 'MG',
 'DO',
 'BI',
 'GD',
 'TT',
 'CG',
 'GM',
 'AG',
 'HN',
 'TG',
 'JM',

In [94]:
df_plot

country,year,US,CN,GB,DE,CA,AU,IT,FR,ES,...,VA,KI,AS,GI,FM,TC,SB,VC,NU,TV
0,1970,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1971,0.167421,0.757576,0.276382,0.366412,0.338889,0.507246,0.532710,0.520325,0.821053,...,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
2,1972,0.155251,0.666667,0.278049,0.404110,0.312849,0.462069,0.537736,0.537879,0.622642,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1973,0.152466,0.672131,0.204878,0.360759,0.303191,0.425806,0.481818,0.446970,0.678571,...,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN
4,1974,0.165217,0.590909,0.221154,0.370370,0.234043,0.450000,0.440367,0.478873,0.689655,...,NaN,NaN,NaN,NaN,NaN,NaN,0.00,NaN,NaN,NaN
5,1975,0.106195,0.567901,0.240566,0.363636,0.250000,0.390244,0.486957,0.426752,0.731343,...,NaN,NaN,NaN,1.000000,NaN,NaN,1.00,NaN,0.0,NaN
6,1976,0.137339,0.600000,0.254545,0.262195,0.234375,0.403509,0.415254,0.318471,0.619718,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,1977,0.120690,0.696970,0.186916,0.297143,0.243655,0.311377,0.401515,0.318750,0.631579,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,1978,0.100437,0.643564,0.197248,0.267442,0.196970,0.350575,0.392857,0.337423,0.632911,...,0.000000,NaN,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,NaN
9,1979,0.135593,0.594059,0.211712,0.219101,0.220588,0.312139,0.330935,0.304878,0.548780,...,NaN,0.000000,1.000000,NaN,NaN,NaN,1.00,NaN,NaN,NaN


In [127]:
dist = "w1"
mode = "individual"
country_list = (
    df_stats
    .query("year == 2022 and mode == @mode")
    .sort_values("total_articles", ascending=False)
    .country
    .head(10)
).to_list()
df_plot = (
    df_pct.query("dist == @dist and mode == @mode")
    .pivot(index="year", columns="country", values="diag")
    [country_list]
)
uf.plotly_heatmap(df_plot,
                  x_labels=df_plot.columns,
                  y_labels=df_plot.index,
                  z_min=0, z_max=0.2,
                  x_type="country",
                  colorscale="non",
                  line_height=10,
                  title=f"{dist} {mode}",
                  )

In [126]:
# dist = "jaccard"
# mode = "individual"
# country_list = (
#     df_stats
#     .query("year == 2022 and mode == @mode")
#     .sort_values("total_articles", ascending=False)
#     .country
#     .head(50)
# ).to_list()
# df_plot = (
#     df_stats.query("mode == @mode")
#     .pivot(index="year", columns="country", values="log_total_articles")
#     [country_list]
# )
# uf.plotly_heatmap(df_plot,
#                   x_labels=df_plot.columns,
#                   y_labels=df_plot.index,
#                   # z_min=0, z_max=0.2,
#                   x_type="country",
#                   colorscale="non",
#                   line_height=10,
#                   title=f"Total articles {mode}",
#                   )

In [125]:
country_list = ["CH", "IR", "ID", "CN"]
px.line(
    df_stats.query("country == @country_list and mode == @mode"),
    x="year",
    y="log_total_articles",
    color="country"
)